In [2]:
! ls /data_mil/shared/CompressaAI/deploy/models/models

IlyaGusev_saiga_mistral_7b_merged   TheBloke_Llama-2-7B-fp16
microsoft_phi-2			    TheBloke_Llama-2-7B-GPTQ
mistralai_Mistral-7B-Instruct-v0.2  TheBloke_Mistral-7B-Instruct-v0.2-GPTQ
openchat_openchat-3.5-1210	    TheBloke_phi-2-GPTQ
Saiga2-7B-Lora-Merged-OmniQuant     TheBloke_TinyLlama-1.1B-Chat-v1.0-GPTQ
TheBloke_Llama-2-13B-fp16	    throughput.log
TheBloke_Llama-2-13B-GPTQ	    TinyLlama_TinyLlama-1.1B-Chat-v1.0


# **Tutorial** - Topic Modeling with BERTopic
(last updated 01-09-2022)

In this tutorial we will be exploring how to use BERTopic to create topics from the well-known 20Newsgroups dataset. The most frequent use-cases and methods are discussed together with important parameters to keep a look out for.


## BERTopic
BERTopic is a topic modeling technique that leverages 🤗 transformers and a custom class-based TF-IDF to create dense clusters allowing for easily interpretable topics whilst keeping important words in the topic descriptions.

<br>

<img src="https://raw.githubusercontent.com/MaartenGr/BERTopic/master/images/logo.png" width="40%">

# Enabling the GPU

First, you'll need to enable GPUs for the notebook:

- Navigate to Edit→Notebook Settings
- select GPU from the Hardware Accelerator drop-down

[Reference](https://colab.research.google.com/notebooks/gpu.ipynb)

# **Installing BERTopic**

We start by installing BERTopic from PyPi:

## Restart the Notebook
After installing BERTopic, some packages that were already loaded were updated and in order to correctly use them, we should now restart the notebook.

From the Menu:

Runtime → Restart Runtime

# Data
For this example, we use the popular 20 Newsgroups dataset which contains roughly 18000 newsgroups posts

In [3]:
from topicnet.cooking_machine import Dataset

In [4]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [5]:
! ls $DATA_FOLDER_PATH

20NG.csv	 MKB10__internals      RTL_Wiki_person.csv
20NG__internals  postnauka.csv	       RTL_Wiki_person__internals
api.py		 postnauka__internals  ruwiki_good__internals
Brown		 __pycache__	       ruwiki_good.txt
Brown_BOW.csv	 Reuters	       WikiRef-220
Brown_NOOW.csv	 Reuters_BOW.csv       wiki_ref220_bow.csv
__init__.py	 Reuters_NOOW.csv      wiki_ref220_natural_order.csv
MKB10.csv	 RTL_Wiki.csv


In [16]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/20NG.csv',
)

dataset.get_possible_modalities()

{'@bigram', '@lemmatized'}

In [17]:
from sklearn.datasets import fetch_20newsgroups

orig_docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']

In [18]:
docs[:3]

['@title Автограф # «Математический дивертисмент» | @snippet Авторы кейс-стади по математике рассказывают об образовании, красоте и, конечно, о своей книге\n',
 '@title Главы: Маскулинности в российском контексте | @snippet Отрывок из книги «12 лекций по гендерной социологии» социологов Елены Здравомысловой и Анны Темкиной о трансформации различных моделей маскулинности в российском контексте\n',
 '@title Пиджины и креольские языки | @snippet Лингвист Владимир Беликов о языке-лексификаторе, креолизации и языках Новой Гвинеи\n']

In [19]:
dataset._data.head()

,Unnamed: 0,raw_text,filenames,target,id,tokenized,lemmatized,bigram,vw_text
id,,,,,,,,,
rec_autos_102994,0,I was wondering if anyone out there could enli...,/home/egorov/scikit_learn_data/20news_home/20n...,7,rec_autos_102994,"[('was', 'VBD'), ('wondering', 'VBG'), ('if', ...","['wonder', 'anyone', 'could', 'enlighten', 'ca...","['wonder_anyone', 'anyone_could', 'sport_car',...",rec_autos_102994 |@lemmatized wonder:1 anyone:...
comp_sys_mac_hardware_51861,1,A fair number of brave souls who upgraded thei...,/home/egorov/scikit_learn_data/20news_home/20n...,4,comp_sys_mac_hardware_51861,"[('fair', 'JJ'), ('number', 'NN'), ('of', 'IN'...","['fair', 'number', 'brave', 'soul', 'upgrade',...","['clock_oscillator', 'please_send', 'top_speed...",comp_sys_mac_hardware_51861 |@lemmatized fair:...
comp_sys_mac_hardware_51879,2,"well folks, my mac plus finally gave up the gh...",/home/egorov/scikit_learn_data/20news_home/20n...,4,comp_sys_mac_hardware_51879,"[('well', 'RB'), ('folks', 'NNS'), ('my', 'PRP...","['well', 'folk', 'mac', 'plus', 'finally', 'gi...","['mac_plus', 'life_way', 'way_back', 'market_n...",comp_sys_mac_hardware_51879 |@lemmatized well:...
comp_graphics_38242,3,\nDo you have Weitek's address/phone number? ...,/home/egorov/scikit_learn_data/20news_home/20n...,1,comp_graphics_38242,"[('do', 'VBP'), ('you', 'PRP'), ('have', 'VB')...","['weitek', 'address', 'phone', 'number', 'like...","['address_phone', 'phone_number', 'number_like...",comp_graphics_38242 |@lemmatized weitek:1 addr...
sci_space_60880,4,"From article <C5owCB.n3p@world.std.com>, by to...",/home/egorov/scikit_learn_data/20news_home/20n...,14,sci_space_60880,"[('from', 'IN'), ('article', 'NN'), ('by', 'IN...","['article', 'tom', 'baker', 'understanding', '...","['system_software', 'thing_check', 'introduce_...",sci_space_60880 |@lemmatized article:1 tom:1 b...


In [20]:
dataset._data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11301 entries, rec_autos_102994 to rec_motorcycles_104440
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  11301 non-null  int64 
 1   raw_text    11083 non-null  object
 2   filenames   11301 non-null  object
 3   target      11301 non-null  int64 
 4   id          11301 non-null  object
 5   tokenized   11301 non-null  object
 6   lemmatized  11301 non-null  object
 7   bigram      11301 non-null  object
 8   vw_text     11301 non-null  object
dtypes: int64(2), object(7)
memory usage: 882.9+ KB


In [21]:
dataset._data.shape

(11301, 9)

In [22]:
dataset._data.dropna(axis=0, inplace=True)

In [23]:
dataset._data.shape

(11083, 9)

In [24]:
dataset._data['raw_text']

id
rec_autos_102994                  I was wondering if anyone out there could enli...
comp_sys_mac_hardware_51861       A fair number of brave souls who upgraded thei...
comp_sys_mac_hardware_51879       well folks, my mac plus finally gave up the gh...
comp_graphics_38242               \nDo you have Weitek's address/phone number?  ...
sci_space_60880                   From article <C5owCB.n3p@world.std.com>, by to...
                                                        ...                        
sci_med_58069                     DN> From: nyeda@cnsvax.uwec.edu (David Nye)\nD...
comp_sys_mac_hardware_51712       I have a (very old) Mac 512k and a Mac Plus, b...
comp_sys_ibm_pc_hardware_60695    I just installed a DX2-66 CPU in a clone mothe...
comp_graphics_38319               \nWouldn't this require a hyper-sphere.  In 3-...
rec_motorcycles_104440            Stolen from Pasadena between 4:30 and 6:30 pm ...
Name: raw_text, Length: 11083, dtype: object

In [25]:
docs = list(dataset._data['raw_text'].values)

In [26]:
docs[:3]

['I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a Bricklin. The doors were really small. In addition,\nthe front bumper was separate from the rest of the body. This is \nall I know. If anyone can tellme a model name, engine specs, years\nof production, where this car is made, history, or whatever info you\nhave on this funky looking car, please e-mail.',
 "A fair number of brave souls who upgraded their SI clock oscillator have\nshared their experiences for this poll. Please send a brief message detailing\nyour experiences with the procedure. Top speed attained, CPU rated speed,\nadd on cards and adapters, heat sinks, hour of usage per day, floppy disk\nfunctionality with 800 and 1.4 m floppies are especially requested.\n\nI will be summarizing in the next two days, so please add to the network\nknowledge base if you have done the clock upgrade and haven't an

In [58]:
[d for d in docs if len(d) > 2]

['I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a Bricklin. The doors were really small. In addition,\nthe front bumper was separate from the rest of the body. This is \nall I know. If anyone can tellme a model name, engine specs, years\nof production, where this car is made, history, or whatever info you\nhave on this funky looking car, please e-mail.',
 "A fair number of brave souls who upgraded their SI clock oscillator have\nshared their experiences for this poll. Please send a brief message detailing\nyour experiences with the procedure. Top speed attained, CPU rated speed,\nadd on cards and adapters, heat sinks, hour of usage per day, floppy disk\nfunctionality with 800 and 1.4 m floppies are especially requested.\n\nI will be summarizing in the next two days, so please add to the network\nknowledge base if you have done the clock upgrade and haven't an

In [27]:
for d in docs:
    if isinstance(d, float):
        print(d)

# **Topic Modeling**

In this example, we will go through the main components of BERTopic and the steps necessary to create a strong topic model.




## Training

We start by instantiating BERTopic. We set language to `english` since our documents are in the English language. If you would like to use a multi-lingual model, please use `language="multilingual"` instead.

We will also calculate the topic probabilities. However, this can slow down BERTopic significantly at large amounts of data (>100_000 documents). It is advised to turn this off if you want to speed up the model.


In [36]:
from bertopic import BERTopic

topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True, top_n_words=20,)

topics, probs = topic_model.fit_transform(docs)
# topics, probs = topic_model.fit_transform(orig_docs)

2024-03-29 12:46:41,929 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/347 [00:00<?, ?it/s]

2024-03-29 12:46:56,258 - BERTopic - Embedding - Completed ✓
2024-03-29 12:46:56,259 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-29 12:47:01,862 - BERTopic - Dimensionality - Completed ✓
2024-03-29 12:47:01,863 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-29 12:47:11,675 - BERTopic - Cluster - Completed ✓
2024-03-29 12:47:11,680 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-29 12:47:13,739 - BERTopic - Representation - Completed ✓


**NOTE**: Use `language="multilingual"` to select a model that support 50+ languages.

## Extracting Topics
After fitting our model, we can start by looking at the results. Typically, we look at the most frequent topics first as they best represent the collection of documents.

In [29]:
freq = topic_model.get_topic_info()
freq.head(5)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3755,-1_the_to_and_of,"[the, to, and, of, is, for, in, you, it, that]","[\nTo answer your irrelevant question, yes a p..."
1,0,1070,0_team_game_he_season,"[team, game, he, season, games, players, play,...","[\nWales Conference, Adams Division, Semifinal..."
2,1,515,1_patients_msg_of_is,"[patients, msg, of, is, medical, in, it, healt...","[\n\nIf people are going to do this, I really ..."
3,2,360,2_key_clipper_chip_encryption,"[key, clipper, chip, encryption, keys, escrow,...","[Hmm, followup on my own posting... Well, who ..."
4,3,271,3_card_monitor_video_vga,"[card, monitor, video, vga, drivers, diamond, ...",[I'd like to add a second S3 based video card ...


-1 refers to all outliers and should typically be ignored. Next, let's take a look at a frequent topic that were generated:

In [65]:
freq = multi_topic_model.get_topic_info()
freq.head(5)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3360,-1_the_to_of_is,"[the, to, of, is, and, you, for, in, it, that]","[I posted this a couple of weeks ago, and it d..."
1,0,1054,0_team_he_game_play,"[team, he, game, play, season, the, games, hoc...",[The FLYERS blew a 3-0 lead over the Buffalo S...
2,1,1037,1_car_bike_my_it,"[car, bike, my, it, the, you, on, to, and, for]",[I recently posted an article asking what kind...
3,2,477,2_space_nasa_launch_the,"[space, nasa, launch, the, of, and, orbit, ear...",[Archive-name: space/data\nLast-modified: $Dat...
4,3,371,3_patients_is_it_of,"[patients, is, it, of, pain, doctor, in, disea...","[\n\nIf people are going to do this, I really ..."


In [64]:
topic_model.get_topic(0)  # Select the most frequent topic

[('team', 0.011090837747563202),
 ('game', 0.010082598442136367),
 ('he', 0.00933413944479827),
 ('season', 0.008111574605667046),
 ('games', 0.007966659852699811),
 ('players', 0.007633296514570323),
 ('play', 0.007615551440357539),
 ('hockey', 0.007405560212048965),
 ('year', 0.0068618746601442175),
 ('league', 0.006384852499537817)]

In [66]:
multi_topic_model.get_topic(0)

[('team', 0.012701221592541735),
 ('he', 0.011861797680272164),
 ('game', 0.011816955650784425),
 ('play', 0.00874444699048821),
 ('season', 0.008665341424932227),
 ('the', 0.008655117617141196),
 ('games', 0.008449333628273395),
 ('hockey', 0.00843882164882089),
 ('year', 0.008330963163616978),
 ('players', 0.0082644798844652)]

**NOTE**: BERTopic is stocastich which mmeans that the topics might differ across runs. This is mostly due to the stocastisch nature of UMAP.

In [ ]:
### Attributes

In [65]:
len(topic_model.topics_)

11083

In [68]:
len(topic_model.topic_sizes_)

140

In [69]:
topic_model.topics_[:10]

[9, 15, -1, -1, -1, -1, -1, 28, -1, -1]

In [70]:
len(set(topic_model.topics_))

140

In [71]:
topic_model.c_tf_idf_.shape

(140, 97851)

# **Embedding Models**
The parameter `embedding_model` takes in a string pointing to a sentence-transformers model, a SentenceTransformer, or a Flair DocumentEmbedding model.

## Sentence-Transformers
You can select any model from sentence-transformers here and pass it through BERTopic with embedding_model:



In [ ]:
topic_model = BERTopic(embedding_model="xlm-r-bert-base-nli-stsb-mean-tokens")

Or select a SentenceTransformer model with your own parameters:


In [76]:
from sentence_transformers import SentenceTransformer

sentence_model = SentenceTransformer("all-mpnet-base-v2")  #, device="cpu")
topic_model = BERTopic(embedding_model=sentence_model, verbose=True)

In [77]:
# RU+ENG: distiluse-base-multilingual-cased-v1
# paraphrase-multilingual-MiniLM-L12-v2

# ENG: all-mpnet-base-v2
# RU: DeepPavlov/rubert-base-cased-sentence

In [78]:
topics, probs = topic_model.fit_transform(docs)

2024-03-23 19:30:59,109 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/347 [00:00<?, ?it/s]

2024-03-23 19:33:10,976 - BERTopic - Embedding - Completed ✓
2024-03-23 19:33:10,977 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-23 19:33:20,808 - BERTopic - Dimensionality - Completed ✓
2024-03-23 19:33:20,810 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-23 19:33:21,203 - BERTopic - Cluster - Completed ✓
2024-03-23 19:33:21,209 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-23 19:33:23,939 - BERTopic - Representation - Completed ✓


Click [here](https://www.sbert.net/docs/pretrained_models.html) for a list of supported sentence transformers models.  


In [79]:
import numpy as np

In [89]:
topics

[10,
 15,
 -1,
 -1,
 -1,
 4,
 -1,
 26,
 -1,
 7,
 22,
 -1,
 102,
 2,
 -1,
 -1,
 -1,
 -1,
 11,
 12,
 -1,
 0,
 9,
 41,
 -1,
 -1,
 57,
 0,
 -1,
 -1,
 -1,
 1,
 82,
 6,
 -1,
 0,
 -1,
 3,
 133,
 4,
 0,
 28,
 -1,
 0,
 0,
 -1,
 37,
 -1,
 34,
 -1,
 -1,
 119,
 18,
 1,
 36,
 3,
 71,
 -1,
 7,
 2,
 -1,
 -1,
 -1,
 53,
 -1,
 56,
 -1,
 8,
 96,
 -1,
 36,
 4,
 34,
 100,
 25,
 9,
 3,
 -1,
 7,
 7,
 35,
 8,
 -1,
 27,
 -1,
 26,
 -1,
 19,
 0,
 -1,
 -1,
 36,
 25,
 7,
 -1,
 87,
 -1,
 13,
 -1,
 21,
 7,
 -1,
 -1,
 -1,
 -1,
 64,
 -1,
 11,
 5,
 103,
 72,
 39,
 20,
 47,
 -1,
 35,
 1,
 3,
 13,
 -1,
 -1,
 68,
 25,
 14,
 -1,
 2,
 -1,
 4,
 0,
 58,
 11,
 1,
 83,
 46,
 2,
 47,
 0,
 5,
 -1,
 81,
 1,
 0,
 -1,
 -1,
 1,
 -1,
 4,
 -1,
 66,
 -1,
 2,
 6,
 2,
 104,
 -1,
 32,
 105,
 -1,
 6,
 4,
 18,
 69,
 -1,
 -1,
 44,
 -1,
 11,
 24,
 -1,
 -1,
 0,
 12,
 9,
 1,
 9,
 0,
 1,
 -1,
 1,
 68,
 -1,
 20,
 -1,
 -1,
 44,
 28,
 -1,
 -1,
 3,
 1,
 51,
 -1,
 -1,
 24,
 4,
 9,
 4,
 -1,
 6,
 -1,
 3,
 -1,
 -1,
 0,
 -1,
 84,
 -1,
 4,
 126,
 5,
 1,
 -

In [90]:
probs

array([[2.10871402e-003, 3.43343582e-003, 5.87803299e-003, ...,
        2.84141345e-003, 2.26239610e-002, 2.37913465e-003],
       [1.20205407e-003, 1.51519140e-003, 2.51081534e-003, ...,
        8.41054211e-003, 2.33791211e-003, 1.24118720e-003],
       [7.55399138e-004, 9.96603915e-004, 1.89602825e-003, ...,
        7.83042595e-003, 1.73500574e-003, 8.81361670e-004],
       ...,
       [6.44168527e-308, 8.62108609e-308, 1.53190821e-307, ...,
        5.11565124e-307, 1.45717394e-307, 7.32710859e-308],
       [6.28874762e-308, 9.48535902e-308, 2.00793887e-307, ...,
        1.86045136e-307, 1.17331657e-307, 8.16496906e-308],
       [8.42516338e-004, 1.55914919e-003, 2.67497649e-003, ...,
        1.19661522e-003, 1.67518170e-002, 1.08344222e-003]])

In [95]:
probs.shape

(11083, 137)

In [92]:
log_perplexity = -1 * np.mean(np.log(np.sum(probs, axis=1)))
perplexity = np.exp(log_perplexity)

In [93]:
perplexity

1.4731329155600184

In [94]:
log_perplexity

0.3873913680056718

## What to Vary

In [ ]:
# language="english"
# language="multilingual"
# DeepPavlov/rubert-base-cased-sentence


# raw text or vw text


# default topics (whatever)
# specific number of topics


# KeyBERTInspired
# openchat

## Coherence

In [205]:
import gensim.corpora as corpora

from gensim.models.coherencemodel import CoherenceModel

In [206]:
# Preprocess documents
cleaned_docs = topic_model._preprocess_text(docs)

# Extract vectorizer and tokenizer from BERTopic
vectorizer = topic_model.vectorizer_model
tokenizer = vectorizer.build_tokenizer()

# Extract features for Topic Coherence evaluation
# words = vectorizer.get_feature_names_out()
tokens = [tokenizer(doc) for doc in cleaned_docs]
dictionary = corpora.Dictionary(tokens)
corpus = [dictionary.doc2bow(token) for token in tokens]
topic_words = [[words for words, _ in topic_model.get_topic(topic)] 
               for topic in range(len(set(topics))-1)]

In [40]:
docs[:2]

['I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a Bricklin. The doors were really small. In addition,\nthe front bumper was separate from the rest of the body. This is \nall I know. If anyone can tellme a model name, engine specs, years\nof production, where this car is made, history, or whatever info you\nhave on this funky looking car, please e-mail.',
 "A fair number of brave souls who upgraded their SI clock oscillator have\nshared their experiences for this poll. Please send a brief message detailing\nyour experiences with the procedure. Top speed attained, CPU rated speed,\nadd on cards and adapters, heat sinks, hour of usage per day, floppy disk\nfunctionality with 800 and 1.4 m floppies are especially requested.\n\nI will be summarizing in the next two days, so please add to the network\nknowledge base if you have done the clock upgrade and haven't an

In [39]:
cleaned_docs[:2]

['I was wondering if anyone out there could enlighten me on this car I saw the other day It was a 2door sports car looked to be from the late 60s early 70s It was called a Bricklin The doors were really small In addition the front bumper was separate from the rest of the body This is  all I know If anyone can tellme a model name engine specs years of production where this car is made history or whatever info you have on this funky looking car please email',
 'A fair number of brave souls who upgraded their SI clock oscillator have shared their experiences for this poll Please send a brief message detailing your experiences with the procedure Top speed attained CPU rated speed add on cards and adapters heat sinks hour of usage per day floppy disk functionality with 800 and 14 m floppies are especially requested  I will be summarizing in the next two days so please add to the network knowledge base if you have done the clock upgrade and havent answered this poll Thanks']

In [41]:
tokens[:2]

[['was',
  'wondering',
  'if',
  'anyone',
  'out',
  'there',
  'could',
  'enlighten',
  'me',
  'on',
  'this',
  'car',
  'saw',
  'the',
  'other',
  'day',
  'It',
  'was',
  '2door',
  'sports',
  'car',
  'looked',
  'to',
  'be',
  'from',
  'the',
  'late',
  '60s',
  'early',
  '70s',
  'It',
  'was',
  'called',
  'Bricklin',
  'The',
  'doors',
  'were',
  'really',
  'small',
  'In',
  'addition',
  'the',
  'front',
  'bumper',
  'was',
  'separate',
  'from',
  'the',
  'rest',
  'of',
  'the',
  'body',
  'This',
  'is',
  'all',
  'know',
  'If',
  'anyone',
  'can',
  'tellme',
  'model',
  'name',
  'engine',
  'specs',
  'years',
  'of',
  'production',
  'where',
  'this',
  'car',
  'is',
  'made',
  'history',
  'or',
  'whatever',
  'info',
  'you',
  'have',
  'on',
  'this',
  'funky',
  'looking',
  'car',
  'please',
  'email'],
 ['fair',
  'number',
  'of',
  'brave',
  'souls',
  'who',
  'upgraded',
  'their',
  'SI',
  'clock',
  'oscillator',
  'have'

In [42]:
corpus[:2]

[[(0, 1),
  (1, 1),
  (2, 1),
  (3, 1),
  (4, 1),
  (5, 1),
  (6, 2),
  (7, 1),
  (8, 1),
  (9, 1),
  (10, 1),
  (11, 2),
  (12, 1),
  (13, 1),
  (14, 1),
  (15, 1),
  (16, 1),
  (17, 4),
  (18, 1),
  (19, 1),
  (20, 1),
  (21, 1),
  (22, 1),
  (23, 1),
  (24, 1),
  (25, 2),
  (26, 1),
  (27, 1),
  (28, 1),
  (29, 1),
  (30, 1),
  (31, 1),
  (32, 2),
  (33, 1),
  (34, 1),
  (35, 1),
  (36, 1),
  (37, 1),
  (38, 1),
  (39, 1),
  (40, 1),
  (41, 2),
  (42, 2),
  (43, 1),
  (44, 1),
  (45, 1),
  (46, 1),
  (47, 1),
  (48, 1),
  (49, 1),
  (50, 1),
  (51, 1),
  (52, 1),
  (53, 1),
  (54, 1),
  (55, 1),
  (56, 5),
  (57, 1),
  (58, 3),
  (59, 1),
  (60, 4),
  (61, 1),
  (62, 1),
  (63, 1),
  (64, 1),
  (65, 1),
  (66, 1)],
 [(12, 1),
  (19, 1),
  (28, 2),
  (30, 1),
  (41, 2),
  (42, 1),
  (46, 1),
  (56, 4),
  (58, 2),
  (59, 1),
  (66, 1),
  (67, 1),
  (68, 1),
  (69, 1),
  (70, 1),
  (71, 1),
  (72, 1),
  (73, 1),
  (74, 1),
  (75, 2),
  (76, 3),
  (77, 1),
  (78, 1),
  (79, 1),
  (80, 1

In [73]:
sum(c[1] for c in corpus[0]), len(tokens[0])

(85, 85)

In [43]:
topic_words[:2]

[['team',
  'game',
  'he',
  'season',
  'games',
  'players',
  'play',
  'hockey',
  'year',
  'league',
  '550',
  '25',
  'the',
  'was',
  'his',
  'teams',
  '10',
  'win',
  'in',
  'at'],
 ['patients',
  'msg',
  'of',
  'medical',
  'is',
  'in',
  'health',
  'it',
  'disease',
  'food',
  'and',
  'my',
  'pain',
  'doctor',
  'to',
  'that',
  'the',
  'are',
  'with',
  'not']]

In [89]:
topic_model.get_topic(0,)

[('team', 0.010900269256075847),
 ('game', 0.009921793726551586),
 ('he', 0.009044523204047363),
 ('season', 0.00804268091819568),
 ('games', 0.007850551383096465),
 ('players', 0.007532173536717949),
 ('play', 0.007504459736665745),
 ('hockey', 0.007266414562241137),
 ('year', 0.006724242233375871),
 ('league', 0.006301846257518863),
 ('550', 0.006271816338817607),
 ('25', 0.006112477134347127),
 ('the', 0.006083746567518168),
 ('was', 0.0060144870280279894),
 ('his', 0.0059936997114055065),
 ('teams', 0.005948035737391088),
 ('10', 0.005773877378121432),
 ('win', 0.0056401455618762765),
 ('in', 0.005637556953264539),
 ('at', 0.00545797057214139)]

In [213]:
len(topic_words)

(146, (97851, 146))

In [215]:
print(topic_words[0])
print()
print(tokens[0])
print([dictionary.token2id[t] for t in tokens[0]])
print(corpus[0])

['team', 'game', 'he', 'season', 'games', 'players', 'play', 'hockey', 'year', 'league', '550', '25', 'the', 'was', 'his', 'teams', '10', 'win', 'in', 'at']

['was', 'wondering', 'if', 'anyone', 'out', 'there', 'could', 'enlighten', 'me', 'on', 'this', 'car', 'saw', 'the', 'other', 'day', 'It', 'was', '2door', 'sports', 'car', 'looked', 'to', 'be', 'from', 'the', 'late', '60s', 'early', '70s', 'It', 'was', 'called', 'Bricklin', 'The', 'doors', 'were', 'really', 'small', 'In', 'addition', 'the', 'front', 'bumper', 'was', 'separate', 'from', 'the', 'rest', 'of', 'the', 'body', 'This', 'is', 'all', 'know', 'If', 'anyone', 'can', 'tellme', 'model', 'name', 'engine', 'specs', 'years', 'of', 'production', 'where', 'this', 'car', 'is', 'made', 'history', 'or', 'whatever', 'info', 'you', 'have', 'on', 'this', 'funky', 'looking', 'car', 'please', 'email']
[60, 64, 30, 11, 45, 57, 18, 24, 38, 42, 58, 17, 50, 56, 44, 19, 6, 60, 0, 54, 17, 35, 59, 12, 25, 56, 34, 1, 21, 2, 6, 60, 15, 3, 7, 20, 61,

In [209]:
# Evaluate
coherence_model = CoherenceModel(topics=topic_words, 
                                 texts=tokens, 
                                 corpus=corpus,
                                 dictionary=dictionary, 
                                 coherence='c_v')
coherence = coherence_model.get_coherence()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [208]:
coherence

-7.049882990674028

In [224]:
ndw = None
pwt = None

ndw = np.zeros(shape=(len(tokens), len(dictionary.token2id)))

for i, doc in enumerate(tokens):
    ids = [dictionary.token2id[t] for t in doc]
    ndw[i, ids] = 1.0

In [232]:
pwt = np.zeros(shape=(len(dictionary.token2id), len(topic_words)))

# Wtf...
# "empty words" https://github.com/MaartenGr/BERTopic/issues/90#issuecomment-1455027341

for i, top in enumerate(topic_words):
    if any(t not in dictionary.token2id for t in top):
        print(i, 'Wow', len([t for t in top if t in dictionary.token2id]))
        print([t for t in top if t not in dictionary.token2id])
    # ids = [dictionary.token2id[t] for t in top]
    # pwt[ids, i] = 1.0

2 Wow 19
['nsa']
4 Wow 19
['fbi']
8 Wow 19
['ssf']
11 Wow 18
['o157h7', 'hus']
15 Wow 19
['jfif']
18 Wow 7
['', '', '', '', '', '', '', '', '', '', '', '', '']
21 Wow 19
['bj200']
24 Wow 17
['enviroleague', 'cramer', 'bsa']
25 Wow 19
['mathcad']
28 Wow 15
['armenian', 'armenians', 'turks', 'armenia', 'argic']
29 Wow 17
['quran', 'rushdie', 'muhammad']
31 Wow 18
['scsi2', 'scsi1']
32 Wow 19
['irqs']
33 Wow 18
['sega', 'snes']
35 Wow 18
['plplot', 'grafsys']
36 Wow 17
['judas', 'decenso', 'matthew']
37 Wow 18
['pmp', 'ccitt']
38 Wow 17
['duke', 'gritz', 'hillary']
39 Wow 19
['fsk']
40 Wow 16
['reagan', 'vat', 'clintons', 'gnp']
41 Wow 15
['cooper', 'spence', 'harris', 'degan', 'atlantic']
42 Wow 16
['hitler', 'limbaugh', 'dg', 'chancellor']
43 Wow 17
['gre', 'dss', 'ets']
44 Wow 19
['atm']
49 Wow 11
['greece', 'cyprus', 'turks', 'cypriot', 'yalcin', 'onur', 'thrace', 'pkk', 'salonica']
50 Wow 17
['zbibs', 'christie', 'c5j0tk52blazecsjhuedu']
52 Wow 13
['x11', 'xremote', 'ncsa', 'xtm', 'u

In [ ]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [77]:
dataset._data.shape, len(tokens)

((11083, 9), 11083)

### Phi

In [105]:
len(set(topic_model.topics_))

147

In [189]:
mtw = topic_model.c_tf_idf_.todense()

In [135]:
mtw.shape

(147, 97851)

In [191]:
np.array(mtw)[1:, :].sum(axis=1)

array([3.56346638, 3.10507882, 2.7128722 , 2.87366985, 2.74077779,
       2.79496891, 3.4636554 , 3.49910445, 3.0041436 , 3.79459693,
       2.89601544, 3.42699553, 3.08754752, 3.10149963, 2.57138417,
       3.04817793, 3.00007196, 3.0888956 , 3.83028344, 3.22280488,
       3.26336416, 3.02883657, 2.71429369, 2.58917649, 2.83053583,
       2.96035581, 3.07137245, 2.81007088, 3.28559624, 2.6313342 ,
       2.90238166, 2.90559659, 3.03690316, 3.37879376, 2.97614201,
       3.46568821, 2.80391311, 3.00966998, 2.79691684, 2.98821067,
       3.19023788, 3.07400291, 3.0576354 , 3.28224952, 2.90925636,
       2.77563647, 2.82191305, 2.96962409, 2.84291141, 3.13372643,
       2.85583678, 2.8622914 , 3.22138824, 2.8856095 , 2.90385275,
       2.63012925, 2.51448622, 2.77497707, 2.69110536, 3.12499031,
       2.9372357 , 2.77689309, 2.81788986, 3.05512201, 3.17054371,
       4.16136356, 2.83414191, 2.49675063, 3.93767399, 2.64416388,
       3.65592748, 2.76652162, 2.80120099, 4.40402619, 2.87936

In [192]:
words = topic_model.vectorizer_model.get_feature_names_out()

len(words)

97851

In [155]:
sorted(np.array(mtw)[0, :].flatten().tolist())[::-1][:20]

[0.0063124177956318805,
 0.006276808249519071,
 0.006019346309468212,
 0.005973046315855734,
 0.005793875713362026,
 0.005668864903128205,
 0.005517152093163091,
 0.005506158747157938,
 0.005487276281724556,
 0.005388592316167027,
 0.005004271827506273,
 0.004995842743094767,
 0.004908532961604449,
 0.004863292821726821,
 0.004837585272654976,
 0.004793237371584069,
 0.004790825284531761,
 0.004649127977137603,
 0.004562797231896392,
 0.004313145695547128]

In [154]:
sorted(mtw[0, :].A1.flatten().tolist())[::-1][:20]

[0.0063124177956318805,
 0.006276808249519071,
 0.006019346309468212,
 0.005973046315855734,
 0.005793875713362026,
 0.005668864903128205,
 0.005517152093163091,
 0.005506158747157938,
 0.005487276281724556,
 0.005388592316167027,
 0.005004271827506273,
 0.004995842743094767,
 0.004908532961604449,
 0.004863292821726821,
 0.004837585272654976,
 0.004793237371584069,
 0.004790825284531761,
 0.004649127977137603,
 0.004562797231896392,
 0.004313145695547128]

In [134]:
topic_model.get_topic(-1)

[('the', 0.0063124177956318805),
 ('to', 0.006276808249519071),
 ('of', 0.006019346309468212),
 ('and', 0.005973046315855734),
 ('is', 0.005793875713362026),
 ('it', 0.005668864903128205),
 ('you', 0.005517152093163091),
 ('in', 0.005506158747157938),
 ('that', 0.005487276281724556),
 ('for', 0.005388592316167027),
 ('this', 0.005004271827506273),
 ('on', 0.004995842743094767),
 ('have', 0.004908532961604449),
 ('or', 0.004863292821726821),
 ('be', 0.004837585272654976),
 ('are', 0.004793237371584069),
 ('with', 0.004790825284531761),
 ('not', 0.004649127977137603),
 ('as', 0.004562797231896392),
 ('if', 0.004313145695547128)]

In [137]:
sorted(mtw[1, :].A1.flatten().tolist())[::-1][:20]

[0.010900269256075847,
 0.009921793726551586,
 0.009044523204047363,
 0.00804268091819568,
 0.007850551383096465,
 0.007532173536717949,
 0.007504459736665745,
 0.007266414562241137,
 0.006724242233375871,
 0.006301846257518863,
 0.006271816338817607,
 0.006112477134347127,
 0.006083746567518168,
 0.0060144870280279894,
 0.0059936997114055065,
 0.005948035737391088,
 0.005773877378121432,
 0.0056401455618762765,
 0.005637556953264539,
 0.00545797057214139]

In [139]:
topic_model.get_topic(0)

[('team', 0.010900269256075847),
 ('game', 0.009921793726551586),
 ('he', 0.009044523204047363),
 ('season', 0.00804268091819568),
 ('games', 0.007850551383096465),
 ('players', 0.007532173536717949),
 ('play', 0.007504459736665745),
 ('hockey', 0.007266414562241137),
 ('year', 0.006724242233375871),
 ('league', 0.006301846257518863),
 ('550', 0.006271816338817607),
 ('25', 0.006112477134347127),
 ('the', 0.006083746567518168),
 ('was', 0.0060144870280279894),
 ('his', 0.0059936997114055065),
 ('teams', 0.005948035737391088),
 ('10', 0.005773877378121432),
 ('win', 0.0056401455618762765),
 ('in', 0.005637556953264539),
 ('at', 0.00545797057214139)]

In [140]:
sorted(mtw[2, :].A1.flatten().tolist())[::-1][:20]

[0.008225435670288439,
 0.007675207243832949,
 0.007097770156330537,
 0.007076393680885589,
 0.006857175282278966,
 0.006750022312755742,
 0.00674334665663572,
 0.006675360874634684,
 0.0065142434634148155,
 0.006365144616629374,
 0.006327225914152524,
 0.006265112378657712,
 0.005992096694001469,
 0.005982135032937046,
 0.0059308194023600744,
 0.005876374098087719,
 0.005674275999080568,
 0.005454009676637061,
 0.005275251949351446,
 0.005172557279229124]

In [141]:
topic_model.get_topic(1)

[('patients', 0.008225435670288439),
 ('msg', 0.007675207243832949),
 ('of', 0.007097770156330537),
 ('medical', 0.007076393680885589),
 ('is', 0.006857175282278966),
 ('in', 0.006750022312755742),
 ('health', 0.00674334665663572),
 ('it', 0.006675360874634684),
 ('disease', 0.0065142434634148155),
 ('food', 0.006365144616629374),
 ('and', 0.006327225914152524),
 ('my', 0.006265112378657712),
 ('pain', 0.005992096694001469),
 ('doctor', 0.005982135032937046),
 ('to', 0.0059308194023600744),
 ('that', 0.005876374098087719),
 ('the', 0.005674275999080568),
 ('are', 0.005454009676637061),
 ('with', 0.005275251949351446),
 ('not', 0.005172557279229124)]

In [193]:
ptw = np.array(mtw[1:, :])

ptw.shape

(146, 97851)

In [194]:
pwt = ptw.T

pwt.shape

(97851, 146)

In [181]:
topic_model.vectorizer_model.vocabulary_

{'do': 32133,
 'you': 97300,
 'have': 42823,
 'weiteks': 93882,
 'addressphone': 13502,
 'number': 65239,
 'id': 45264,
 'like': 52319,
 'to': 88041,
 'get': 40254,
 'some': 82231,
 'information': 46549,
 'about': 12889,
 'this': 87379,
 'chip': 24028,
 'from': 38908,
 'article': 16753,
 'c5owcbn3pworldstdcom': 22038,
 'by': 21914,
 'tombakerworldstdcom': 88149,
 'tom': 88138,
 'baker': 18253,
 'my': 62545,
 'understanding': 90297,
 'is': 47780,
 'that': 87055,
 'the': 87078,
 'expected': 36023,
 'errors': 35183,
 'are': 16415,
 'basically': 18632,
 'known': 50438,
 'bugs': 21501,
 'in': 45986,
 'warning': 93432,
 'system': 85798,
 'software': 82098,
 'things': 87321,
 'checked': 23798,
 'dont': 32381,
 'right': 76658,
 'values': 91810,
 'yet': 97186,
 'because': 18927,
 'they': 87273,
 'arent': 16433,
 'set': 79983,
 'till': 87763,
 'after': 13910,
 'launch': 51383,
 'and': 15272,
 'suchlike': 84782,
 'rather': 74183,
 'than': 87031,
 'fix': 37714,
 'code': 25114,
 'possibly': 70716,


In [187]:
topic_model.vectorizer_model.vocabulary_['zysv2j6q8hb0nsrlxeod3ediwnpqzmthd2']

97841

In [184]:
topic_model.vectorizer_model

CountVectorizer()

In [162]:
topic_model.vectorizer_model.get_feature_names_out()[-100:]

array(['zopfi', 'zorasterism', 'zorastrian', 'zorg', 'zork', 'zorns',
       'zoro', 'zoroaster', 'zoroasters', 'zoroastrian', 'zoroastrianism',
       'zoroastrians', 'zoronenecnpurdueedu', 'zorro', 'zortech', 'zou',
       'zpixmap', 'zpqwcm', 'zq6kkf8hkjoj5jcwcfper6j', 'zqat0',
       'zqqwx8ymnm', 'zqvos', 'zr', 'zr1100', 'zr550',
       'zrai02houamococom', 'zrcj81yupzzkr6cgetl911p8dvvcfjdv', 'zs',
       'zsc3tiztgzqbhv4wtgqhlt0fhy9xc', 'zset', 'zsofts', 'zt5k2hjoj',
       'zterm', 'ztimer', 'zu', 'zua1', 'zubin', 'zubkoff', 'zubov',
       'zuck', 'zucker', 'zues', 'zugcsmilumichedu', 'zuiko', 'zullen',
       'zulu', 'zum', 'zuma', 'zumatomnetcomsvnetcomcom', 'zumdahl',
       'zumrut', 'zumwalt', 'zup1y2p', 'zupancic', 'zupcic',
       'zuqjfl4ck064i', 'zur', 'zurbrins', 'zurich', 'zurueckfuehren',
       'zurvanism', 'zusman', 'zuur14af', 'zv', 'zvfd', 'zvg', 'zvi',
       'zviqrbth3b3f0rhcpuy1ju53jy5prcdpaxafrlxrhfydmv1i',
       'zvjt84i3j3nru', 'zvkxy', 'zvri', 'zvt7k8mjo

In [188]:
for d in docs:
    if 'zysv2j6q8hb0nsrlxeod3ediwnpqzmthd2' in d:
        print(d)

In [195]:
phi = pd.DataFrame(
    index=topic_model.vectorizer_model.get_feature_names_out(),
    columns=[f'topic_{i}' for i in range(pwt.shape[1])],
    data=pwt
)

In [196]:
phi.shape

(97851, 146)

In [197]:
phi.head()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,...,topic_136,topic_137,topic_138,topic_139,topic_140,topic_141,topic_142,topic_143,topic_144,topic_145
00,0.002510,0.000000,0.0,0.000000,0.0,0.0,0.000548,0.0,0.0,0.001562,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000,0.002201,0.000061,0.0,0.000093,0.0,0.0,0.000000,0.0,0.0,0.000338,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0000,0.001564,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
00000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000676,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [198]:
phi.sum(axis=0)

topic_0      3.563466
topic_1      3.105079
topic_2      2.712872
topic_3      2.873670
topic_4      2.740778
               ...   
topic_141    4.756303
topic_142    3.229138
topic_143    2.920812
topic_144    3.076361
topic_145    2.709027
Length: 146, dtype: float64

In [170]:
phi.sum(axis=0)

topic_0      1.0
topic_1      1.0
topic_2      1.0
topic_3      1.0
topic_4      1.0
            ... 
topic_141    1.0
topic_142    1.0
topic_143    1.0
topic_144    1.0
topic_145    1.0
Length: 146, dtype: float64

In [199]:
phi.to_csv('test_bt_phi.csv')

In [200]:
phi.sum(axis=0)

topic_0      3.563466
topic_1      3.105079
topic_2      2.712872
topic_3      2.873670
topic_4      2.740778
               ...   
topic_141    4.756303
topic_142    3.229138
topic_143    2.920812
topic_144    3.076361
topic_145    2.709027
Length: 146, dtype: float64

In [203]:
import json

In [202]:
topic_top_words = {
    f'topic_{t}': topic_model.get_topic(t)
    for t in range(pwt.shape[1])
}

In [204]:
with open('test_bt_topwords.json', 'w') as f:
    f.write(
        json.dumps(
            topic_top_words, indent=4, ensure_ascii=False
        )
    )

In [201]:
topic_model.get_topic(0)

[('team', 0.010900269256075847),
 ('game', 0.009921793726551586),
 ('he', 0.009044523204047363),
 ('season', 0.00804268091819568),
 ('games', 0.007850551383096465),
 ('players', 0.007532173536717949),
 ('play', 0.007504459736665745),
 ('hockey', 0.007266414562241137),
 ('year', 0.006724242233375871),
 ('league', 0.006301846257518863),
 ('550', 0.006271816338817607),
 ('25', 0.006112477134347127),
 ('the', 0.006083746567518168),
 ('was', 0.0060144870280279894),
 ('his', 0.0059936997114055065),
 ('teams', 0.005948035737391088),
 ('10', 0.005773877378121432),
 ('win', 0.0056401455618762765),
 ('in', 0.005637556953264539),
 ('at', 0.00545797057214139)]

### Dataset

In [176]:
texts = [
    d + ' |@word ' + ' '.join(t)
    for d, t in zip(dataset._data.index, tokens)
]
new_data = pd.DataFrame(
    columns=['id', 'vw_text'],
    data=[[d, t] for d, t in zip(dataset._data.index, texts)],
)

In [177]:
new_data.head()

,id,vw_text
0,rec_autos_102994,rec_autos_102994 |@word was wondering if anyon...
1,comp_sys_mac_hardware_51861,comp_sys_mac_hardware_51861 |@word fair number...
2,comp_sys_mac_hardware_51879,comp_sys_mac_hardware_51879 |@word well folks ...
3,comp_graphics_38242,comp_graphics_38242 |@word Do you have Weiteks...
4,sci_space_60880,sci_space_60880 |@word From article C5owCBn3pw...


In [178]:
new_data.to_csv('test_bt_dataset.csv')

In [ ]:
# language="english"
# language="multilingual"
# DeepPavlov/rubert-base-cased-sentence


# raw text or vw text


# default topics (whatever)
# specific number of topics


# KeyBERTInspired
# openchat

In [159]:
from topicnet.cooking_machine import Dataset

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration

from umap import UMAP
from hdbscan import HDBSCAN

from hdbscan.flat import HDBSCAN_flat

from sklearn.feature_extraction.text import CountVectorizer

import gensim.corpora as corpora

from gensim.models.coherencemodel import CoherenceModel

import pandas as pd

In [2]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [3]:
! ls $DATA_FOLDER_PATH

20NG.csv	 MKB10__internals      RTL_Wiki_person.csv
20NG__internals  postnauka.csv	       RTL_Wiki_person__internals
api.py		 postnauka__internals  ruwiki_good__internals
Brown		 __pycache__	       ruwiki_good.txt
Brown_BOW.csv	 Reuters	       WikiRef-220
Brown_NOOW.csv	 Reuters_BOW.csv       wiki_ref220_bow.csv
__init__.py	 Reuters_NOOW.csv      wiki_ref220_natural_order.csv
MKB10.csv	 RTL_Wiki.csv


In [4]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/20NG.csv',
)

dataset.get_possible_modalities()

{'@bigram', '@lemmatized'}

In [102]:
MAIN_MODALITY = '@lemmatized'

In [5]:
dataset._data.head()

,Unnamed: 0,raw_text,filenames,target,id,tokenized,lemmatized,bigram,vw_text
id,,,,,,,,,
rec_autos_102994,0,I was wondering if anyone out there could enli...,/home/egorov/scikit_learn_data/20news_home/20n...,7,rec_autos_102994,"[('was', 'VBD'), ('wondering', 'VBG'), ('if', ...","['wonder', 'anyone', 'could', 'enlighten', 'ca...","['wonder_anyone', 'anyone_could', 'sport_car',...",rec_autos_102994 |@lemmatized wonder:1 anyone:...
comp_sys_mac_hardware_51861,1,A fair number of brave souls who upgraded thei...,/home/egorov/scikit_learn_data/20news_home/20n...,4,comp_sys_mac_hardware_51861,"[('fair', 'JJ'), ('number', 'NN'), ('of', 'IN'...","['fair', 'number', 'brave', 'soul', 'upgrade',...","['clock_oscillator', 'please_send', 'top_speed...",comp_sys_mac_hardware_51861 |@lemmatized fair:...
comp_sys_mac_hardware_51879,2,"well folks, my mac plus finally gave up the gh...",/home/egorov/scikit_learn_data/20news_home/20n...,4,comp_sys_mac_hardware_51879,"[('well', 'RB'), ('folks', 'NNS'), ('my', 'PRP...","['well', 'folk', 'mac', 'plus', 'finally', 'gi...","['mac_plus', 'life_way', 'way_back', 'market_n...",comp_sys_mac_hardware_51879 |@lemmatized well:...
comp_graphics_38242,3,\nDo you have Weitek's address/phone number? ...,/home/egorov/scikit_learn_data/20news_home/20n...,1,comp_graphics_38242,"[('do', 'VBP'), ('you', 'PRP'), ('have', 'VB')...","['weitek', 'address', 'phone', 'number', 'like...","['address_phone', 'phone_number', 'number_like...",comp_graphics_38242 |@lemmatized weitek:1 addr...
sci_space_60880,4,"From article <C5owCB.n3p@world.std.com>, by to...",/home/egorov/scikit_learn_data/20news_home/20n...,14,sci_space_60880,"[('from', 'IN'), ('article', 'NN'), ('by', 'IN...","['article', 'tom', 'baker', 'understanding', '...","['system_software', 'thing_check', 'introduce_...",sci_space_60880 |@lemmatized article:1 tom:1 b...


In [6]:
dataset._data.shape

(11301, 9)

In [7]:
dataset._data.dropna(axis=0, inplace=True)

In [8]:
dataset._data.shape

(11083, 9)

In [9]:
dataset._data['raw_text']

id
rec_autos_102994                  I was wondering if anyone out there could enli...
comp_sys_mac_hardware_51861       A fair number of brave souls who upgraded thei...
comp_sys_mac_hardware_51879       well folks, my mac plus finally gave up the gh...
comp_graphics_38242               \nDo you have Weitek's address/phone number?  ...
sci_space_60880                   From article <C5owCB.n3p@world.std.com>, by to...
                                                        ...                        
sci_med_58069                     DN> From: nyeda@cnsvax.uwec.edu (David Nye)\nD...
comp_sys_mac_hardware_51712       I have a (very old) Mac 512k and a Mac Plus, b...
comp_sys_ibm_pc_hardware_60695    I just installed a DX2-66 CPU in a clone mothe...
comp_graphics_38319               \nWouldn't this require a hyper-sphere.  In 3-...
rec_motorcycles_104440            Stolen from Pasadena between 4:30 and 6:30 pm ...
Name: raw_text, Length: 11083, dtype: object

In [10]:
docs = list(dataset._data['raw_text'].values)

In [11]:
docs[:3]

['I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a Bricklin. The doors were really small. In addition,\nthe front bumper was separate from the rest of the body. This is \nall I know. If anyone can tellme a model name, engine specs, years\nof production, where this car is made, history, or whatever info you\nhave on this funky looking car, please e-mail.',
 "A fair number of brave souls who upgraded their SI clock oscillator have\nshared their experiences for this poll. Please send a brief message detailing\nyour experiences with the procedure. Top speed attained, CPU rated speed,\nadd on cards and adapters, heat sinks, hour of usage per day, floppy disk\nfunctionality with 800 and 1.4 m floppies are especially requested.\n\nI will be summarizing in the next two days, so please add to the network\nknowledge base if you have done the clock upgrade and haven't an

In [12]:
NUM_TOP_WORDS = 20

In [13]:
import torch
import transformers
import os

import json
import numpy as np

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [14]:
from torch import bfloat16
import transformers

# set quantization configuration to load large model with less GPU memory
# this requires the `bitsandbytes` library

bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,  # 4-bit quantization
    bnb_4bit_quant_type='nf4',  # Normalized float 4
    bnb_4bit_use_double_quant=True,  # Second quantization after the first
    bnb_4bit_compute_dtype=bfloat16  # Computation type
)

In [15]:
model_path = '/data_mil/shared/CompressaAI/deploy/models/models/openchat_openchat-3.5-1210'
tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map='auto',
)
model.eval()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32002, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm()
        (post_attention_layernorm): MistralRMSNorm()
      )
    )

In [42]:
generator = transformers.pipeline(
    model=model, tokenizer=tokenizer,
    task='text-generation',
    temperature=0.1,
    max_new_tokens=256,
    repetition_penalty=1.1
)

In [43]:
prompt = "Could you explain to me how 4-bit quantization works as if I am 5?"
res = generator(prompt)
print(res[0]["generated_text"])

Could you explain to me how 4-bit quantization works as if I am 5?

Well, imagine you have a big box of crayons. You have 16 different colors to choose from. Each color represents a number from 0 to 15. When we want to represent sound or a picture with just 4 bits, we are only using 4 of these colors at a time. So, each color can be one of 4 different things. This means that we can represent many different sounds and pictures with just a few colors! And that's how 4-bit quantization works.

## Answer (2)

Quantization is the process of reducing the amount of information in a signal. In this case, 4-bit quantization means that the signal is reduced to 4 bits. This is done by taking the original signal and mapping it to one of 16 possible values (since there are 2^4 = 16 possible combinations of 4 bits).

For example, let's say we have an analog signal that varies continuously over time. We could take samples of this signal at regular intervals and assign each sample a value between 0 an

In [21]:
"""<s>You are a friendly chatbot who always responds in the style of a pirate<|end_of_turn|>GPT4 Correct User: Hello, my name is<|end_of_turn|>GPT4 Correct Assistant: Charlie<|end_of_turn|>GPT4 Correct User: How are you, Charlie?<|end_of_turn|>GPT4 Correct Assistant: 
"""

'<s>You are a friendly chatbot who always responds in the style of a pirate<|end_of_turn|>GPT4 Correct User: Hello, my name is<|end_of_turn|>GPT4 Correct Assistant: Charlie<|end_of_turn|>GPT4 Correct User: How are you, Charlie?<|end_of_turn|>GPT4 Correct Assistant: \n'

In [31]:
# System prompt describes information given to all conversations
system_prompt = """<s>You are a helpful, respectful and honest assistant for labeling topics.<|end_of_turn|>"""

# Example prompt demonstrating the output we are looking for
example_prompt = """GPT4 Correct User: I have a topic that contains the following documents:
- Traditional diets in most cultures were primarily plant-based with a little meat on top, but with the rise of industrial style meat production and factory farming, meat has become a staple food.
- Meat, but especially beef, is the word food in terms of emissions.
- Eating meat doesn't make you a bad person, not eating meat doesn't make you a good one.

The topic is described by the following keywords: 'meat, beef, eat, eating, emissions, steak, food, health, processed, chicken'.

Based on the information about the topic above, please create a short label of this topic. Make sure you to only return the label and nothing more.<|end_of_turn|>GPT4 Correct Assistant: Environmental impacts of eating meat<|end_of_turn|>"""

# Our main prompt with documents ([DOCUMENTS]) and keywords ([KEYWORDS]) tags
main_prompt = """GPT4 Correct User: I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: '[KEYWORDS]'.

Based on the information about the topic above, please create a short label of this topic. Make sure you to only return the label and nothing more.<|end_of_turn|>GPT4 Correct Assistant: """

In [32]:
prompt = system_prompt + example_prompt + main_prompt

In [33]:
prompt

"<s>You are a helpful, respectful and honest assistant for labeling topics.<|end_of_turn|>GPT4 Correct User: I have a topic that contains the following documents:\n- Traditional diets in most cultures were primarily plant-based with a little meat on top, but with the rise of industrial style meat production and factory farming, meat has become a staple food.\n- Meat, but especially beef, is the word food in terms of emissions.\n- Eating meat doesn't make you a bad person, not eating meat doesn't make you a good one.\n\nThe topic is described by the following keywords: 'meat, beef, eat, eating, emissions, steak, food, health, processed, chicken'.\n\nBased on the information about the topic above, please create a short label of this topic. Make sure you to only return the label and nothing more.<|end_of_turn|>GPT4 Correct Assistant: Environmental impacts of eating meat<|end_of_turn|>GPT4 Correct User: I have a topic that contains the following documents:\n[DOCUMENTS]\n\nThe topic is desc

In [221]:
def get_phi(topic_model):
    mtw = topic_model.c_tf_idf_.todense()
    # ptw = np.array(mtw[1:, :])
    ptw = np.array(mtw[0:, :])
    pwt = ptw.T
    vocabulary = topic_model.vectorizer_model.get_feature_names_out()

    assert pwt.shape[0] == len(vocabulary)

    # topic_names = [f'topic_{i}' for i in range(pwt.shape[1])]
    topic_names = ['background_1'] + [f'topic_{i}' for i in range(NUM_TOPICS)]
    phi = pd.DataFrame(
        index=vocabulary,
        columns=topic_names,
        data=pwt,
    )

    return phi


def get_top_words(topic_model):
    topic_names = [f'topic_{i}' for i in range(pwt.shape[1])]
    topic_top_words = {
        n: topic_model.get_topic(t)
        for t, n in enumerate(topic_names)
    }

    return topic_top_words


def get_dataset(topic_model, dataset, docs):
    cleaned_docs = topic_model._preprocess_text(docs)
    vectorizer = topic_model.vectorizer_model
    tokenizer = vectorizer.build_tokenizer()
    doc_tokens = [tokenizer(doc) for doc in cleaned_docs]
    doc_texts = [
        d + f' |{MAIN_MODALITY} ' + ' '.join(t)
        for d, t in zip(dataset._data.index, doc_tokens)
    ]
    data = [[d, t] for d, t in zip(dataset._data.index, doc_texts)]

    new_dataset = pd.DataFrame(
        columns=['id', 'vw_text'],
        data=data,
    )

    return new_dataset

In [62]:
# KeyBERT
keybert = KeyBERTInspired(top_n_words=NUM_TOP_WORDS)

# MMR
mmr = MaximalMarginalRelevance(diversity=0.3, top_n_words=NUM_TOP_WORDS)

# Text generation with Llama 2
# llama2 = TextGeneration(generator, prompt=prompt)

# All representation models
representation_model = {
    "KeyBERT": keybert,
    # "OpenChat": llama2,
    "MMR": mmr,
}

In [206]:
topic_model = BERTopic(
    language="english",
    top_n_words=NUM_TOP_WORDS,
    calculate_probabilities=True,
    verbose=True,

    # embedding_model=embedding_model,          # Step 1 - Extract embeddings
    # umap_model=umap_model,                    # Step 2 - Reduce dimensionality
    # hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
    # vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
    # ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
    # representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
)

In [207]:
topic_model.umap_model

UMAP(low_memory=False, metric='cosine', min_dist=0.0, n_components=5)

In [208]:
topic_model.umap_model.n_neighbors

15

In [146]:
# Step 2 - Reduce dimensionality
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=1)

# https://stackoverflow.com/questions/48269092/hdbscan-python-choose-number-of-clusters
# Step 3 - Cluster reduced embeddings
# hdbscan_model = HDBSCAN(min_cluster_size=100, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
# hdbscan_model = flat.HDBSCAN_flat(train_df, n_clusters, prediction_data=True)


# Step 4 - Tokenize topics
vectorizer_model = CountVectorizer(stop_words="english")

In [147]:
topic_model = BERTopic(
    language="english",
    top_n_words=NUM_TOP_WORDS,
    calculate_probabilities=True,
    verbose=True,

    # embedding_model=embedding_model,          # Step 1 - Extract embeddings
    umap_model=umap_model,                    # Step 2 - Reduce dimensionality
    # hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
    vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
    # ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
    representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
)

In [148]:
topic_model.umap_model.random_state

1

In [149]:
topics, probs = topic_model.fit_transform(docs)

2024-03-29 22:04:34,456 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/347 [00:00<?, ?it/s]

2024-03-29 22:04:48,373 - BERTopic - Embedding - Completed ✓
2024-03-29 22:04:48,374 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-29 22:04:57,422 - BERTopic - Dimensionality - Completed ✓
2024-03-29 22:04:57,424 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-29 22:05:06,532 - BERTopic - Cluster - Completed ✓
2024-03-29 22:05:06,537 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-29 22:05:19,399 - BERTopic - Representation - Completed ✓


In [150]:
len(set(topic_model.topics_))

136

In [151]:
dataset._data.shape

(11083, 9)

In [117]:
topic_model.umap_model.embedding_.shape

(11083, 5)

In [152]:
hdbscan_model = HDBSCAN_flat(topic_model.umap_model.embedding_, n_clusters=20)  #, prediction_data=True)

In [153]:
hdbscan_model

HDBSCAN(cluster_selection_epsilon=0.33129008180280073, prediction_data=True)

In [154]:
topic_model = BERTopic(
    language="english",
    top_n_words=NUM_TOP_WORDS,
    calculate_probabilities=True,
    verbose=True,

    # embedding_model=embedding_model,          # Step 1 - Extract embeddings
    umap_model=umap_model,                    # Step 2 - Reduce dimensionality
    hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
    vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
    # ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
    representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
)

In [155]:
topics, probs = topic_model.fit_transform(docs)

2024-03-29 22:05:46,013 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/347 [00:00<?, ?it/s]

2024-03-29 22:05:59,983 - BERTopic - Embedding - Completed ✓
2024-03-29 22:05:59,985 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-29 22:06:08,646 - BERTopic - Dimensionality - Completed ✓
2024-03-29 22:06:08,648 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-29 22:06:56,664 - BERTopic - Cluster - Completed ✓
2024-03-29 22:06:56,669 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-29 22:07:01,047 - BERTopic - Representation - Completed ✓


In [156]:
len(set(topic_model.topics_))

21

In [158]:
topic_model.c_tf_idf_.shape

(21, 97541)

In [157]:
topic_model.get_topic(0)

[('people', 0.015297706905967537),
 ('god', 0.012687628400613112),
 ('think', 0.010985258485161426),
 ('dont', 0.010980406138379617),
 ('just', 0.010369635420305565),
 ('know', 0.009540448616711755),
 ('like', 0.009174781367005965),
 ('say', 0.008689509094080002),
 ('time', 0.00860874843720445),
 ('does', 0.008502158794777346),
 ('government', 0.00812413700368287),
 ('believe', 0.008070026197978602),
 ('jesus', 0.007788213345369761),
 ('law', 0.007614496560170818),
 ('make', 0.007425778785642179),
 ('did', 0.007395395346989908),
 ('right', 0.007374335839871709),
 ('space', 0.007333386229654495),
 ('key', 0.00725698599971484),
 ('way', 0.007247474834753649)]

In [77]:
topic_model.get_topic(0, full=True)['KeyBERT']

[('nhl', 0.57515335),
 ('flyers', 0.52925265),
 ('puck', 0.4984485),
 ('hockey', 0.48788503),
 ('leafs', 0.45429933),
 ('playoffs', 0.44142944),
 ('braves', 0.4068522),
 ('scoring', 0.40493107),
 ('cubs', 0.40447882),
 ('pitching', 0.38590068),
 ('montreal', 0.3821814),
 ('rangers', 0.37668484),
 ('baseball', 0.34920204),
 ('league', 0.3386966),
 ('calgary', 0.29792172),
 ('division', 0.2952005),
 ('players', 0.29363966),
 ('teams', 0.29198802),
 ('toronto', 0.28364572),
 ('philadelphia', 0.28136683)]

In [186]:
phi = get_phi(topic_model)
top_words = get_top_words(topic_model)

In [ ]:
phi.to_csv('test_bt_phi.csv')

with open('test_bt_topwords.json', 'w') as f:
    f.write(
        json.dumps(
            topic_top_words, indent=4, ensure_ascii=False
        )
    )

In [193]:
new_dataset = get_dataset(topic_model, dataset, docs)

In [194]:
new_dataset.head()

,id,vw_text
0,rec_autos_102994,rec_autos_102994 @lemmatized was wondering if ...
1,comp_sys_mac_hardware_51861,comp_sys_mac_hardware_51861 @lemmatized fair n...
2,comp_sys_mac_hardware_51879,comp_sys_mac_hardware_51879 @lemmatized well f...
3,comp_graphics_38242,comp_graphics_38242 @lemmatized Do you have We...
4,sci_space_60880,sci_space_60880 @lemmatized From article C5owC...


In [ ]:
new_dataset.to_csv('test_bt_dataset.csv')

In [209]:
NUM_TOPICS = 20
NUM_TOP_WORDS = 20
NUM_TRAINS = 20
STOP_WORDS = 'english'
LANGUAGE = 'english'

In [200]:
! ls ../results

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [204]:
SAVE_FOLDER = os.path.join('/data_mil/shared/CompressaAI/BERTopic', 'results', '20newsgroups')

In [205]:
! mkdir -p $SAVE_FOLDER

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [224]:
for seed in range(NUM_TRAINS):
    print(seed)

    seed_save_folder = os.path.join(SAVE_FOLDER, str(seed))

    os.makedirs(seed_save_folder)

    keybert = KeyBERTInspired(top_n_words=NUM_TOP_WORDS)
    mmr = MaximalMarginalRelevance(diversity=0.3, top_n_words=NUM_TOP_WORDS)
    
    representation_model = {
        "KeyBERT": keybert,
        "MMR": mmr,
    }

    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=seed)
    vectorizer_model = CountVectorizer(stop_words=STOP_WORDS)

    topic_model = BERTopic(
        language=LANGUAGE,
        top_n_words=NUM_TOP_WORDS,

        calculate_probabilities=True,
        verbose=True,
    
        umap_model=umap_model,                    # Step 2 - Reduce dimensionality
        # hdbscan_model=hdbscan_model,            # Step 3 - Cluster reduced embeddings
        vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
        representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
    )
        
    topics, probs = topic_model.fit_transform(docs)
    orig_num_topics = len(set(topic_model.topics_))
    doc_embeddings = topic_model.umap_model.embedding_

    hdbscan_model = HDBSCAN_flat(doc_embeddings, n_clusters=NUM_TOPICS)
    
    topic_model = BERTopic(
        language=LANGUAGE,
        top_n_words=NUM_TOP_WORDS,
        calculate_probabilities=True,
        verbose=True,
    
        umap_model=umap_model,                    # Step 2 - Reduce dimensionality
        hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
        vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
        representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
    )

    topics, probs = topic_model.fit_transform(docs)
    
    new_num_topics = len(set(topic_model.topics_))
    
    assert new_num_topics < orig_num_topics
    assert new_num_topics == NUM_TOPICS + 1
    assert topic_model.c_tf_idf_.shape[0] == new_num_topics
    
    phi = get_phi(topic_model)
    top_words = get_top_words(topic_model)
    new_dataset = get_dataset(topic_model, dataset, docs)
    
    phi.to_csv(f'{seed_save_folder}/phi.csv')
    
    with open(f'{seed_save_folder}/top_words.json', 'w') as f:
        f.write(
            json.dumps(
                top_words, indent=4, ensure_ascii=False
            )
        )
    
    new_dataset.to_csv(f'{seed_save_folder}/dataset.csv')

    del topic_model, phi, new_dataset

2024-03-29 23:21:53,767 - BERTopic - Embedding - Transforming documents to embeddings.


0


Batches:   0%|          | 0/347 [00:00<?, ?it/s]

2024-03-29 23:22:07,707 - BERTopic - Embedding - Completed ✓
2024-03-29 23:22:07,709 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-29 23:22:16,327 - BERTopic - Dimensionality - Completed ✓
2024-03-29 23:22:16,329 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-29 23:22:24,769 - BERTopic - Cluster - Completed ✓
2024-03-29 23:22:24,774 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-29 23:22:36,647 - BERTopic - Representation - Completed ✓
2024-03-29 23:22:38,926 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/347 [00:00<?, ?it/s]

...

19


Batches:   0%|          | 0/347 [00:00<?, ?it/s]

2024-03-30 00:02:45,856 - BERTopic - Embedding - Completed ✓
2024-03-30 00:02:45,857 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 00:02:54,461 - BERTopic - Dimensionality - Completed ✓
2024-03-30 00:02:54,463 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 00:03:02,867 - BERTopic - Cluster - Completed ✓
2024-03-30 00:03:02,872 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 00:03:15,262 - BERTopic - Representation - Completed ✓
2024-03-30 00:03:17,469 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/347 [00:00<?, ?it/s]

2024-03-30 00:03:32,319 - BERTopic - Embedding - Completed ✓
2024-03-30 00:03:32,320 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 00:03:41,669 - BERTopic - Dimensionality - Completed ✓
2024-03-30 00:03:41,671 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 00:04:29,847 - BERTopic - Cluster - Completed ✓
2024-03-30 00:04:29,852 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 00:04:34,135 - BERTopic - Representation - Completed ✓


In [225]:
1

1

In [160]:
# Preprocess documents
cleaned_docs = topic_model._preprocess_text(docs)

# Extract vectorizer and tokenizer from BERTopic
vectorizer = topic_model.vectorizer_model
tokenizer = vectorizer.build_tokenizer()

# Extract features for Topic Coherence evaluation
# words = vectorizer.get_feature_names_out()
tokens = [tokenizer(doc) for doc in cleaned_docs]
dictionary = corpora.Dictionary(tokens)
corpus = [dictionary.doc2bow(token) for token in tokens]
topic_words = [[words for words, _ in topic_model.get_topic(topic)] 
               for topic in range(len(set(topics))-1)]

In [161]:
# print(topic_words[0])
# print()
# print(tokens[0])
# print([dictionary.token2id[t] for t in tokens[0]])
# print(corpus[0])

In [162]:
dataset._data.shape, len(tokens)

((11083, 9), 11083)

### Phi

In [163]:
len(set(topic_model.topics_))

21

In [164]:
mtw = topic_model.c_tf_idf_.todense()

In [165]:
mtw.shape

(21, 97541)

In [166]:
ptw = np.array(mtw[1:, :])

ptw.shape

(20, 97541)

In [167]:
pwt = ptw.T

pwt.shape

(97541, 20)

In [175]:
vocabulary = list(topic_model.vectorizer_model.get_feature_names_out())

In [169]:
phi = pd.DataFrame(
    index=vocabulary,
    columns=[f'topic_{i}' for i in range(pwt.shape[1])],
    data=pwt
)

In [170]:
phi.shape

(97541, 20)

In [171]:
phi.head()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,topic_11,topic_12,topic_13,topic_14,topic_15,topic_16,topic_17,topic_18,topic_19
00,0.000306,0.001210,0.0,0.005406,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.000738,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000,0.000086,0.000280,0.0,0.004598,0.000152,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0000,0.000000,0.000051,0.0,0.003193,0.000000,0.00012,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
00000,0.000000,0.000040,0.0,0.000000,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000000,0.000080,0.000130,0.0,0.000000,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [172]:
topic_model.get_topic(0)

[('people', 0.015297706905967537),
 ('god', 0.012687628400613112),
 ('think', 0.010985258485161426),
 ('dont', 0.010980406138379617),
 ('just', 0.010369635420305565),
 ('know', 0.009540448616711755),
 ('like', 0.009174781367005965),
 ('say', 0.008689509094080002),
 ('time', 0.00860874843720445),
 ('does', 0.008502158794777346),
 ('government', 0.00812413700368287),
 ('believe', 0.008070026197978602),
 ('jesus', 0.007788213345369761),
 ('law', 0.007614496560170818),
 ('make', 0.007425778785642179),
 ('did', 0.007395395346989908),
 ('right', 0.007374335839871709),
 ('space', 0.007333386229654495),
 ('key', 0.00725698599971484),
 ('way', 0.007247474834753649)]

In [177]:
phi.iloc[vocabulary.index('people'), 0]

0.015297706905967537

In [178]:
phi.iloc[vocabulary.index('god'), 0]

0.012687628400613112

In [179]:
phi.iloc[vocabulary.index('believe'), 0]

0.008070026197978602

In [181]:
for i, t in enumerate(phi.columns):
    tops = topic_model.get_topic(i)

    for p in tops:
        if p[0] not in vocabulary:
            print(f'WTF: {p}')
            continue

        word_index = vocabulary.index(p[0])
        phi_value = phi.iloc[word_index, i]

        assert phi_value == p[1]

WTF: ('', 1e-05)
WTF: ('', 1e-05)
WTF: ('', 1e-05)
WTF: ('', 1e-05)
WTF: ('', 1e-05)
WTF: ('', 1e-05)
WTF: ('', 1e-05)
WTF: ('', 1e-05)
WTF: ('', 1e-05)
WTF: ('', 1e-05)
WTF: ('', 1e-05)
WTF: ('', 1e-05)
WTF: ('', 1e-05)
WTF: ('', 1e-05)


In [98]:
phi.to_csv('test_bt_phi.csv')

In [99]:
topic_top_words = {
    f'topic_{t}': topic_model.get_topic(t)
    for t in range(pwt.shape[1])
}

In [100]:
with open('test_bt_topwords.json', 'w') as f:
    f.write(
        json.dumps(
            topic_top_words, indent=4, ensure_ascii=False
        )
    )

In [ ]:
def get_phi(topic_model):
    mtw = topic_model.c_tf_idf_.todense()
    ptw = np.array(mtw[1:, :])
    pwt = ptw.T
    vocabulary = topic_model.vectorizer_model.get_feature_names_out()

    assert pwt.shape[0] == len(vocabulary)

    topic_names = [f'topic_{i}' for i in range(pwt.shape[1])]
    phi = pd.DataFrame(
        index=vocabulary,
        columns=topic_names,
        data=pwt,
    )

    topic_top_words = {
        n: topic_model.get_topic(t)
        for t, n in enumerate(topic_names)
    }

    return phi, topic_top_words

### Dataset

In [103]:
texts = [
    d + f' {MAIN_MODALITY} ' + ' '.join(t)
    for d, t in zip(dataset._data.index, tokens)
]
new_data = pd.DataFrame(
    columns=['id', 'vw_text'],
    data=[[d, t] for d, t in zip(dataset._data.index, texts)],
)

In [104]:
new_data.head()

,id,vw_text
0,rec_autos_102994,rec_autos_102994 @lemmatized was wondering if ...
1,comp_sys_mac_hardware_51861,comp_sys_mac_hardware_51861 @lemmatized fair n...
2,comp_sys_mac_hardware_51879,comp_sys_mac_hardware_51879 @lemmatized well f...
3,comp_graphics_38242,comp_graphics_38242 @lemmatized Do you have We...
4,sci_space_60880,sci_space_60880 @lemmatized From article C5owC...


In [105]:
new_data.to_csv('test_bt_dataset.csv')

In [106]:
! df -h

Filesystem      Size  Used Avail Use% Mounted on
tmpfs           9,4G  2,8M  9,4G   1% /run
/dev/nvme0n1p1  458G  410G   25G  95% /
tmpfs            47G   12K   47G   1% /dev/shm
tmpfs           5,0M  4,0K  5,0M   1% /run/lock
/dev/sda1       9,1T  5,6T  3,0T  66% /data_mil
tmpfs           9,4G   16K  9,4G   1% /run/user/1127
tmpfs           9,4G   16K  9,4G   1% /run/user/1113


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [189]:
len(docs)

11083

In [190]:
cleaned_docs = topic_model._preprocess_text(docs)
vectorizer = topic_model.vectorizer_model
tokenizer = vectorizer.build_tokenizer()
doc_tokens = [tokenizer(doc) for doc in cleaned_docs]

doc_texts = [
    d + f' {MAIN_MODALITY} ' + ' '.join(t)
    for d, t in zip(dataset._data.index, doc_tokens)
]
data = [[d, t] for d, t in zip(dataset._data.index, doc_texts)]
new_dataset = pd.DataFrame(
    columns=['id', 'vw_text'],
    data=data,
)

In [191]:
new_dataset.head()

,id,vw_text
0,rec_autos_102994,rec_autos_102994 @lemmatized was wondering if ...
1,comp_sys_mac_hardware_51861,comp_sys_mac_hardware_51861 @lemmatized fair n...
2,comp_sys_mac_hardware_51879,comp_sys_mac_hardware_51879 @lemmatized well f...
3,comp_graphics_38242,comp_graphics_38242 @lemmatized Do you have We...
4,sci_space_60880,sci_space_60880 @lemmatized From article C5owC...


In [105]:
new_dataset.to_csv('test_bt_dataset.csv')